<a href="https://colab.research.google.com/github/ArturStachnik/WorkshopPaleoML/blob/main/sesion_3/H3_abrir_la_caja_negra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Abrir la caja negra

**Machine learning y paleoclima · Red Paleonet · Granada, 2026**

En la sesión 2 entrenamos modelos que estiman la temperatura del agua a partir de los foraminíferos del sedimento, y les preguntamos **cuánto**. Aquí les damos la vuelta y les preguntamos **por qué**: qué especies usan, cómo actúa cada una y si lo que cuentan tiene sentido ecológico.

| | Bloque | Qué se hace aquí |
|---|---|---|
| 1 | **Nuestra caja negra** | Entrenar el modelo de la sesión 2 y ver por qué no se puede leer |
| 2 | **Cuánto pesa** | Qué especies usa el modelo |
| 3 | **Cómo actúa** | La curva de efecto de cada especie, frente a su ecología |
| 4 | **El reparto de un sitio** | Por qué ha dicho esa temperatura en un sitio concreto |
| 5 | **¿Cuentan lo mismo?** | Varios modelos distintos, ¿explican igual? |
| 6 | **La trampa** | Lo que pasa si le damos las coordenadas |
| 7 | **Los límites** | Qué pasa fuera de lo que el modelo ha visto |

**No hay que escribir código.** Cada celda tiene un botón de play; las que tienen controles a la derecha (desplegables, deslizadores) se cambian con el ratón y se vuelven a ejecutar.

El código está escondido. Para verlo, doble clic en la celda.

---
## 0. Preparar el cuaderno

**Qué hacer:** ejecutar la celda. Tarda un poco más que en la sesión 2 porque, si hace falta, instala `shap`, la librería que reparte las predicciones entre las variables.

Primero, como siempre: **Archivo → Guardar una copia en Drive**.

In [ ]:
#@title Preparar el cuaderno
import os, sys, json, shutil, tempfile, logging, warnings, subprocess, urllib.request
try:
    import shap
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"], check=True)
    import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.collections import PolyCollection
from IPython.display import display, Markdown
from sklearn.model_selection import KFold, GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from scipy.stats import spearmanr
try:
    from xgboost import XGBRegressor
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"], check=True)
    from xgboost import XGBRegressor
warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
logging.getLogger("shap").setLevel(logging.ERROR)
%config InlineBackend.figure_format = "retina"

REPO = "https://raw.githubusercontent.com/ArturStachnik/WorkshopPaleoML/main/"


def ruta(fichero):
    """El fichero del repositorio: del disco si está descargado, de GitHub si no."""
    for base in ("", "../"):
        if os.path.exists(base + fichero):
            return base + fichero
    return REPO + fichero


def leer_texto(fichero):
    origen = ruta(fichero)
    if origen.startswith("http"):
        return urllib.request.urlopen(origen, timeout=60).read().decode("utf-8")
    return open(origen, encoding="utf-8").read()


# --- tipografía y colores del taller ---
FUENTES = os.path.join(tempfile.gettempdir(), "fuentes_taller")
os.makedirs(FUENTES, exist_ok=True)
for f in ("DMSans.ttf", "DMSans-Italic.ttf", "DMSerifDisplay.ttf"):
    destino = os.path.join(FUENTES, f)
    try:
        if not os.path.exists(destino):
            origen = ruta("recursos/fuentes/" + f)
            if origen.startswith("http"):
                urllib.request.urlretrieve(origen, destino)
            else:
                shutil.copy(origen, destino)
        font_manager.fontManager.addfont(destino)
    except Exception:
        pass

TINTA, GRIS, ARENA, AZUL, ROJO, ORO, VERDE = "#1D2B36", "#8A8F94", "#E8E2D6", "#1F6E8C", "#C2462E", "#D9A441", "#6B8F71"
SST_CMAP = LinearSegmentedColormap.from_list(
    "sst", ["#16405A", AZUL, "#8DB9C9", "#EFE6D6", "#E6A592", ROJO, "#7E2A1A"])
EFECTO = LinearSegmentedColormap.from_list("efecto", ["#16405A", AZUL, "#E9E4DA", ROJO, "#7E2A1A"])
plt.rcParams.update({
    "font.family": ["DM Sans", "DejaVu Sans"], "font.size": 11,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA, "axes.titlesize": 13,
    "axes.titlelocation": "left", "axes.titlecolor": TINTA, "axes.titlepad": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#EEEAE2", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "xtick.color": GRIS, "ytick.color": GRIS, "xtick.labelcolor": TINTA, "ytick.labelcolor": TINTA,
    "figure.dpi": 100, "figure.facecolor": "white", "axes.facecolor": "white",
    "legend.frameon": False, "text.color": TINTA,
})

# --- el mapa ---
_tierra = json.loads(leer_texto("datos/tierra_ne_110m.geojson"))
POLIGONOS = []
for f in _tierra["features"]:
    g = f["geometry"]
    for p in (g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]):
        POLIGONOS.append(np.array(p[0]))


def fondo_mapa(ax, lat=(-80, 90)):
    ax.add_collection(PolyCollection(POLIGONOS, facecolor=ARENA, edgecolor="none", zorder=0))
    ax.set_xlim(-180, 180); ax.set_ylim(*lat); ax.set_aspect("equal")
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_xticks([-120, -60, 0, 60, 120]); ax.set_yticks([-60, -30, 0, 30, 60])
    ax.tick_params(length=0, labelsize=9, labelcolor=GRIS)


def corto(nombre):
    """Neogloboquadrina_pachyderma -> N. pachyderma"""
    p = nombre.split("_")
    return p[0][0] + ". " + " ".join(p[1:])


def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))


def es(v, n=1):
    """Número con coma decimal: 1.15 -> '1,2'"""
    return f"{v:.{n}f}".replace(".", ",")


# --- los datos, ya limpios: el resultado de la sesión 2 ---
datos = pd.read_csv(ruta("datos/forams_sst.csv.gz")).drop_duplicates().reset_index(drop=True)
SITIO = ["Latitude", "Longitude", "Ocean", "sst"]
ESPECIES = [c for c in datos.columns if c not in SITIO]
datos = datos.dropna(subset=["sst", "Latitude", "Longitude"] + ESPECIES).reset_index(drop=True)
USADAS = [e for e in ESPECIES if datos[e].max() * 100 >= 1]          # como en la sesión 2: fuera las raras
X = np.sqrt(datos[USADAS].clip(lower=0))                               # raíz cuadrada (Hellinger)
X.columns = [corto(e) for e in USADAS]
y = datos["sst"].values
CLAVE = [corto(e) for e in ["Neogloboquadrina_pachyderma", "Neogloboquadrina_incompta", "Globoconella_inflata",
                            "Globigerinoides_white", "Trilobatus_sacculifer"]]
ABUND = datos[USADAS].set_axis(X.columns, axis=1) * 100                # abundancia en %
VALIDACION = KFold(5, shuffle=True, random_state=0)


def nuevo_xgboost():
    return XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, n_jobs=2, tree_method="hist",
                        random_state=0)


print(f"Listo. {len(datos)} sitios y {X.shape[1]} especies, como al final de la sesión 2.")

---
# 1. Nuestra caja negra

Partimos del final de la sesión 2: los datos limpios, las 46 especies en raíz cuadrada y un **XGBoost**, uno de los modelos que mejor funcionaba. Aquí lo elegimos por una razón práctica: es de los más rápidos de explicar con SHAP.

In [ ]:
#@title 1.1 Entrenar el modelo y ver cuánto acierta
prediccion = cross_val_predict(nuevo_xgboost(), X, y, cv=VALIDACION)
modelo = nuevo_xgboost().fit(X, y)
n_nodos = len(modelo.get_booster().trees_to_dataframe())

fig, ax = plt.subplots(figsize=(5.6, 5.6))
ax.scatter(y, prediccion, s=6, color=AZUL, alpha=0.4, lw=0)
ax.plot([-4, 32], [-4, 32], color=TINTA, lw=1, ls=(0, (4, 3)))
ax.set_xlim(-4, 32); ax.set_ylim(-4, 32); ax.set_aspect("equal")
ax.set_xlabel("temperatura real (°C)"); ax.set_ylabel("temperatura predicha (°C)")
ax.set_title("XGBoost, con datos que no ha visto")
ax.text(0.04, 0.96, f"error {es(rmse(y, prediccion), 2)} °C", transform=ax.transAxes, va="top", fontsize=13, color=AZUL)
plt.show()
nodos = f"{n_nodos:,}".replace(",", ".")
display(Markdown(
    f"Acierta con un error de **{es(rmse(y, prediccion))} °C** en sitios que no ha visto. "
    f"Por dentro son **300 árboles con {nodos} nodos** en total, cada uno con su pregunta y su umbral. "
    "Todo está a la vista y nada se entiende. Es una caja negra."))

### 1.2 Repartir cada predicción

SHAP hace con cada sitio lo que hicimos con la cuenta del viernes: reparte la temperatura predicha entre las especies, partiendo de la media de todos los sitios. Lo calculamos una vez para todos los sitios y lo usamos en el resto del cuaderno.

In [ ]:
#@title 1.2 Calcular los valores SHAP de todos los sitios
explicador = shap.TreeExplainer(modelo)
SV = explicador.shap_values(X)
BASE = float(np.ravel(explicador.expected_value)[0])
comprobacion = np.abs(BASE + SV.sum(1) - modelo.predict(X)).max()
display(Markdown(
    f"Hecho, **{SV.shape[0]} sitios × {SV.shape[1]} especies**. La media de partida es **{es(BASE)} °C**. "
    f"Y las cuentas cuadran, como las del viernes. En todos los sitios, la media más los aportes de cada especie "
    f"da la predicción del modelo (diferencia máxima {es(comprobacion, 4)} °C)."))

---
# 2. Cuánto pesa

La primera pregunta: **qué especies usa el modelo**. Para cada especie, la media de lo que mueve la predicción, sin importar el signo.

In [ ]:
#@title 2.1 Las especies que más pesan
peso = pd.Series(np.abs(SV).mean(0), index=X.columns).sort_values(ascending=False)
top = peso.head(15)[::-1]
fig, ax = plt.subplots(figsize=(8, 5.6))
ax.barh(top.index, top.values, color=[ROJO if e in CLAVE else GRIS for e in top.index], height=0.65)
ax.set_xlabel("lo que mueve la predicción, de media (°C)")
ax.set_title("Las 15 especies que más pesan en el modelo")
for t in ax.get_yticklabels():
    t.set_style("italic")
ax.grid(axis="y", visible=False)
ax.text(0.98, 0.04, "en rojo, las 5 especies clave\nque elegimos en la sesión 2", transform=ax.transAxes,
        ha="right", va="bottom", color=ROJO, fontsize=11)
plt.show()
cuantas = sum(e in CLAVE for e in peso.index[:5])
display(Markdown(f"De las 5 especies que más pesan, **{cuantas} son las 5 especies clave** que elegimos a mano "
                 "en la sesión 2 mirando sus curvas de respuesta. Nadie se lo ha dicho al modelo."))

### 2.2 El enjambre

La misma información, sin promediar. Cada punto es un sitio: a la derecha, la especie sube la temperatura predicha; a la izquierda, la baja. El color es la abundancia de la especie en ese sitio (rojo, mucha; azul, poca).

**Cómo leerlo:** si los puntos rojos están a la izquierda, la especie abundante indica agua fría. Si están a la derecha, agua cálida.

In [ ]:
#@title 2.2 El enjambre de las 12 especies que más pesan
explicacion = shap.Explanation(values=SV, base_values=np.full(len(SV), BASE), data=X.values,
                               feature_names=list(X.columns))
shap.plots.beeswarm(explicacion, max_display=12, show=False, color=EFECTO, color_bar_label="abundancia")
fig = plt.gcf(); fig.set_size_inches(8.5, 6)
ax = fig.axes[0]
ax.set_xlabel("efecto en la temperatura predicha (°C)")
etiquetas = []
for t in ax.get_yticklabels():
    texto = t.get_text()
    etiquetas.append(f"las otras {X.shape[1] - 11} especies" if texto.startswith("Sum of") else texto)
ax.set_yticks(ax.get_yticks()); ax.set_yticklabels(etiquetas)
for t in ax.get_yticklabels()[1:] if etiquetas[0].startswith("las otras") else ax.get_yticklabels()[:-1]:
    t.set_style("italic")
for cb in fig.axes[1:]:
    cb.set_yticklabels(["poca", "mucha"])
plt.show()

---
# 3. Cómo actúa

Saber cuánto pesa una especie no dice **cómo** actúa. Aquí se pone, al lado, lo que ya sabíamos de la ecología (la curva de respuesta de la sesión 2, abundancia frente a temperatura) y lo que ha aprendido el modelo (el efecto de la especie frente a su abundancia).

**Qué hacer:** elegir una especie en el desplegable y ejecutar.

In [ ]:
#@title 3.1 Una especie: su ecología frente a lo que ha aprendido el modelo
especie = "N. pachyderma"  #@param ["N. pachyderma", "G. inflata", "N. incompta", "G. white", "T. sacculifer", "G. bulloides", "N. dutertrei", "P. obliquiloculata", "G. menardii", "G. truncatulinoides", "G. ruber", "G. glutinata"]
j = list(X.columns).index(especie)
fig, axs = plt.subplots(1, 2, figsize=(13, 4.4), gridspec_kw={"wspace": 0.28})
tramos = np.arange(-2, 31, 2); centro = (tramos[:-1] + tramos[1:]) / 2
axs[0].scatter(y, ABUND[especie], s=4, color=GRIS, alpha=0.3, lw=0)
media = [ABUND.loc[(y >= a) & (y < b), especie].mean() for a, b in zip(tramos[:-1], tramos[1:])]
axs[0].plot(centro, media, color=TINTA, lw=2.8)
axs[0].set_xlabel("temperatura del agua (°C)"); axs[0].set_ylabel("abundancia (%)")
axs[0].set_title(f"Lo que sabíamos: dónde vive {especie}")
axs[1].scatter(ABUND[especie], SV[:, j], s=6, c=y, cmap=SST_CMAP, alpha=0.6, lw=0)
axs[1].axhline(0, color=GRIS, lw=1, ls=(0, (3, 3)))
orden = np.argsort(ABUND[especie].values)
suave = pd.Series(SV[orden, j]).rolling(150, center=True, min_periods=30).mean()
axs[1].plot(ABUND[especie].values[orden], suave, color=TINTA, lw=2.8)
axs[1].set_xlabel(f"abundancia de {especie} (%)"); axs[1].set_ylabel("efecto en la predicción (°C)")
axs[1].set_title("Lo que ha aprendido el modelo")
sm = plt.cm.ScalarMappable(cmap=SST_CMAP, norm=plt.Normalize(y.min(), y.max()))
plt.colorbar(sm, ax=axs[1], shrink=0.85, pad=0.02).set_label("temperatura real (°C)")
plt.show()
r_eco = spearmanr(y, ABUND[especie]).statistic
r_mod = spearmanr(ABUND[especie], SV[:, j]).statistic
coincide = np.sign(r_eco) == np.sign(r_mod)
display(Markdown(
    f"En los datos, cuanto más {'cálida' if r_eco > 0 else 'fría'} el agua, más *{especie}*. "
    f"El modelo ha aprendido que cuanta más *{especie}*, {'más sube' if r_mod > 0 else 'más baja'} la temperatura predicha. "
    + ("**Coinciden.**" if coincide else "**No coinciden: aquí el modelo cuenta algo distinto a la ecología.**")))

### 3.2 Las cinco especies clave de un vistazo

In [ ]:
#@title 3.2 Las curvas de efecto de las cinco especies clave
fig, axs = plt.subplots(1, 5, figsize=(17, 3.6), sharey=True, gridspec_kw={"wspace": 0.1})
for ax, e in zip(axs, CLAVE):
    j = list(X.columns).index(e)
    ax.scatter(ABUND[e], SV[:, j], s=4, c=y, cmap=SST_CMAP, alpha=0.5, lw=0)
    orden = np.argsort(ABUND[e].values)
    ax.plot(ABUND[e].values[orden], pd.Series(SV[orden, j]).rolling(150, center=True, min_periods=30).mean(),
            color=TINTA, lw=2.4)
    ax.axhline(0, color=GRIS, lw=1, ls=(0, (3, 3)))
    ax.set_title(e, style="italic"); ax.set_xlabel("abundancia (%)")
axs[0].set_ylabel("efecto (°C)")
plt.show()

Las de agua fría (*N. pachyderma*, *N. incompta*, *G. inflata*) bajan la temperatura predicha y las tropicales (*G. white*, *T. sacculifer*) la suben. Fíjate también en la forma: casi ninguna es una recta, y varias se saturan. Un poco de especie ya dice mucho; más cantidad añade poco.

---
# 4. El reparto de un sitio

Hasta aquí, el modelo en general. Ahora, **una predicción concreta**: la cascada, como el ticket del viernes 6. Parte de la media de todos los sitios y cada especie suma o resta hasta llegar a la temperatura predicha.

**Qué hacer:** elegir un sitio y ejecutar.

In [ ]:
#@title 4.1 Por qué ha dicho esa temperatura en este sitio
sitio = "Mar de Alborán, cerca de Granada"  #@param ["Mar de Alborán, cerca de Granada", "Mar de Noruega", "Atlántico Norte, frente a Irlanda", "Caribe", "Pacífico ecuatorial", "Mar Arábigo", "Océano Austral"]
LUGARES = {"Mar de Alborán, cerca de Granada": (36.2, -3.3), "Mar de Noruega": (68, 5),
           "Atlántico Norte, frente a Irlanda": (53, -20), "Caribe": (15, -75), "Pacífico ecuatorial": (0, -140),
           "Mar Arábigo": (15, 62), "Océano Austral": (-55, 0)}
la, lo = LUGARES[sitio]
i = int(((datos["Latitude"] - la) ** 2 + (datos["Longitude"] - lo) ** 2).idxmin())
s = SV[i]
o = np.argsort(-np.abs(s))
n = 8
nombres = [f"{X.columns[j]}  ({ABUND.iloc[i, j]:.0f} %)" for j in o[:n]] + [f"las otras {len(s) - n} especies"]
valores = list(s[o[:n]]) + [s[o[n:]].sum()]

fig = plt.figure(figsize=(14, 5.2))
axm = fig.add_axes([0.0, 0.1, 0.3, 0.8]); fondo_mapa(axm)
axm.scatter(datos["Longitude"], datos["Latitude"], s=2, color=GRIS, alpha=0.3, lw=0)
axm.scatter(datos.loc[i, "Longitude"], datos.loc[i, "Latitude"], s=160, marker="*", color=ROJO, edgecolor="white", lw=1)
axm.set_title(sitio)
ax = fig.add_axes([0.42, 0.12, 0.56, 0.76])
acum = BASE
tramo = [BASE]
for k, (nm, v) in enumerate(zip(nombres, valores)):
    yk = len(nombres) - k
    ax.barh(yk, v, left=acum, color=ROJO if v > 0 else AZUL, height=0.62)
    ax.text(max(acum, acum + v) + 0.12 if v > 0 else min(acum, acum + v) - 0.12, yk,
            ("+" if v > 0 else "−") + es(abs(v)), va="center",
            ha="left" if v > 0 else "right", color=ROJO if v > 0 else AZUL, fontsize=11)
    acum += v
    tramo.append(acum)
ancho = max(tramo + [y[i]]) - min(tramo + [y[i]])
ax.set_xlim(min(tramo + [y[i]]) - 0.18 * ancho - 0.5, max(tramo + [y[i]]) + 0.18 * ancho + 0.5)
ax.set_yticks(range(len(nombres), 0, -1)); ax.set_yticklabels(nombres)
for t in ax.get_yticklabels()[:-1]:
    t.set_style("italic")
ax.axvline(BASE, color=GRIS, lw=1, ls=(0, (3, 3)))
ax.axvline(acum, color=TINTA, lw=1.2, ls=(0, (3, 3)))
ax.axvline(y[i], color=VERDE, lw=2)
ax.set_xlabel("temperatura (°C)"); ax.grid(axis="y", visible=False)
ax.set_title(f"media {es(BASE)} °C  →  predicción {es(acum)} °C   ·   real {es(y[i])} °C (línea verde)")
plt.show()
ausentes = [X.columns[j] for j in o[:n] if ABUND.iloc[i, j] < 0.5 and abs(s[j]) > 0.3]
if ausentes:
    display(Markdown("Fíjate en las especies con 0 %: **no estar también informa**. Que no haya *" +
                     "*, *".join(ausentes) + "* ya mueve la predicción."))

### 4.2 Dónde empuja cada especie

Los valores SHAP de una especie en todos los sitios, sobre el mapa. Rojo, la especie sube la temperatura predicha en ese sitio; azul, la baja.

In [ ]:
#@title 4.2 El mapa del efecto de una especie
especie_mapa = "N. pachyderma"  #@param ["N. pachyderma", "G. inflata", "N. incompta", "G. white", "T. sacculifer", "G. bulloides", "N. dutertrei", "G. menardii"]
j = list(X.columns).index(especie_mapa)
lim = np.percentile(np.abs(SV[:, j]), 99)
fig, ax = plt.subplots(figsize=(13, 6.2))
fondo_mapa(ax)
orden = np.argsort(np.abs(SV[:, j]))
sc = ax.scatter(datos["Longitude"].values[orden], datos["Latitude"].values[orden], c=SV[orden, j], s=7,
                cmap=EFECTO, norm=TwoSlopeNorm(0, -lim, lim), lw=0)
plt.colorbar(sc, ax=ax, shrink=0.7, pad=0.01).set_label("efecto en la predicción (°C)")
ax.set_title(f"Dónde empuja {especie_mapa}", style="italic")
plt.show()

---
# 5. ¿Cuentan lo mismo?

La prueba de fuego. Entrenamos **cinco modelos que funcionan de forma muy distinta** (una recta, un bosque, árboles en serie, vecinos y una red neuronal) y le pedimos a SHAP que explique cada uno.

* Si coinciden, la relación es de los datos.
* Si un modelo cuenta algo distinto, la relación es de ese modelo.

Tarda alrededor de un minuto: los vecinos y la red no tienen atajo para SHAP y hay que tantear muchas combinaciones.

In [ ]:
#@title 5.1 Cinco modelos, cinco explicaciones
MODELOS = {
    "Lineal": Ridge(alpha=1.0),
    "Random forest": RandomForestRegressor(n_estimators=100, max_depth=12, n_jobs=-1, random_state=0),
    "XGBoost": nuevo_xgboost(),
    "k-NN (MAT)": KNeighborsRegressor(n_neighbors=5),
    "Red neuronal": make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(32, 32), early_stopping=True,
                                                                 max_iter=500, random_state=0)),
}
rng = np.random.default_rng(0)
muestra = X.iloc[rng.choice(len(X), 150, replace=False)]
fondo = shap.sample(X, 60, random_state=0)
IMP, SENT, ERR = {}, {}, {}
for nombre, m in MODELOS.items():
    ERR[nombre] = rmse(y, cross_val_predict(m, X, y, cv=VALIDACION))
    m.fit(X, y)
    if nombre == "Lineal":
        sv = shap.LinearExplainer(m, fondo).shap_values(muestra)
    elif nombre in ("Random forest", "XGBoost"):
        sv = shap.TreeExplainer(m).shap_values(muestra)
    else:
        sv = shap.PermutationExplainer(m.predict, fondo)(muestra, max_evals=2 * X.shape[1] + 1, silent=True).values
    IMP[nombre] = pd.Series(np.abs(sv).mean(0), index=X.columns)
    SENT[nombre] = pd.Series([np.nan_to_num(spearmanr(muestra.iloc[:, j], sv[:, j]).statistic)
                              for j in range(X.shape[1])], index=X.columns)
    print(f"{nombre}: error {es(ERR[nombre], 2)} °C")
IMP, SENT = pd.DataFrame(IMP), pd.DataFrame(SENT)

In [ ]:
#@title 5.2 La tabla del acuerdo
REL = IMP / IMP.max()
filas = REL.mean(1).sort_values(ascending=False).index[:12]
fig, ax = plt.subplots(figsize=(10, 7.2))
for a, nombre in enumerate(IMP.columns):
    for b, e in enumerate(filas):
        ax.scatter(a, b, s=40 + 900 * REL.loc[e, nombre], color=EFECTO((SENT.loc[e, nombre] + 1) / 2),
                   edgecolor="white", lw=1)
ax.set_xticks(range(len(IMP.columns)))
ax.set_xticklabels([f"{n}\nerror {es(ERR[n])} °C" for n in IMP.columns])
ax.set_yticks(range(len(filas))); ax.set_yticklabels(filas)
for t in ax.get_yticklabels():
    t.set_style("italic")
ax.invert_yaxis(); ax.grid(False); ax.set_xlim(-0.6, len(IMP.columns) - 0.4)
for s in ax.spines.values():
    s.set_visible(False)
ax.tick_params(length=0)
ax.set_title("El tamaño es cuánto pesa la especie en cada modelo. El color, si la toma por especie de agua "
             "cálida (rojo) o fría (azul).", fontsize=12)
plt.show()
mediana = SENT.loc[filas].median(1)
discrepa = {n: [e for e in filas if abs(mediana[e]) > 0.4 and abs(SENT.loc[e, n]) > 0.4
                and np.sign(SENT.loc[e, n]) != np.sign(mediana[e])] for n in IMP.columns}
texto = [f"* La **{n.lower() if n == 'Red neuronal' else n}** (error {es(ERR[n])} °C) cuenta lo contrario que los "
         "demás con *" + "*, *".join(v) + "*." for n, v in discrepa.items() if v]
display(Markdown("**Lectura.** Las filas con el mismo color en todas las columnas son relaciones en las que "
                 "coinciden modelos muy distintos. Esas son de los datos.\n\n" +
                 ("\n".join(texto) if texto else "Todos los modelos coinciden en el sentido de las 12 especies.")))

Un modelo con peor error no es inútil para entender: si coincide con los demás, su explicación también vale. Y un modelo que cuenta una historia distinta a la del resto merece sospecha, aunque sus números no sean malos. **No hace falta el mejor modelo para ver relaciones, pero sí hace falta mirar varios.**

---
# 6. La trampa

En la sesión 2 vimos que, si se le dan al modelo la latitud y la longitud de cada sitio, el error baja. Parece buena idea. SHAP deja ver **en qué se apoya** ese modelo, y para el paleoclima eso lo cambia todo.

In [ ]:
#@title 6.1 Especies más coordenadas: ¿en qué se apoya el modelo?
XC = X.assign(latitud=datos["Latitude"].values, longitud=datos["Longitude"].values)
bloques = (np.floor(datos["Latitude"] / 30) * 100 + np.floor(datos["Longitude"] / 30)).values
err = {}
for nombre, M in (("solo especies", X), ("especies + coordenadas", XC)):
    err[nombre] = (rmse(y, cross_val_predict(nuevo_xgboost(), M, y, cv=VALIDACION)),
                   rmse(y, cross_val_predict(nuevo_xgboost(), M, y, cv=GroupKFold(5), groups=bloques)))
mc = nuevo_xgboost().fit(XC, y)
svc = shap.TreeExplainer(mc).shap_values(XC)
pc = pd.Series(np.abs(svc).mean(0), index=XC.columns).sort_values(ascending=False).head(10)[::-1]

fig, axs = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1.2, 1], "wspace": 0.45})
axs[0].barh(pc.index, pc.values, color=[ROJO if e in ("latitud", "longitud") else GRIS for e in pc.index], height=0.65)
axs[0].set_title("Lo que más pesa con coordenadas"); axs[0].grid(axis="y", visible=False)
axs[0].set_xlabel("lo que mueve la predicción, de media (°C)")
for t in axs[0].get_yticklabels():
    if t.get_text() not in ("latitud", "longitud"):
        t.set_style("italic")
nombres = list(err)
xx = np.arange(2)
axs[1].bar(xx - 0.18, [err[n][0] for n in nombres], 0.34, color=AZUL, label="validación al azar")
axs[1].bar(xx + 0.18, [err[n][1] for n in nombres], 0.34, color=ROJO, label="validación por bloques")
for k, n in enumerate(nombres):
    for dx, v in ((-0.18, err[n][0]), (0.18, err[n][1])):
        axs[1].text(k + dx, v + 0.03, es(v, 2), ha="center", fontsize=11)
axs[1].set_ylim(0, max(max(v) for v in err.values()) * 1.35)
axs[1].set_xticks(xx); axs[1].set_xticklabels(nombres); axs[1].set_ylabel("error (°C)")
axs[1].legend(loc="upper right", ncol=2, fontsize=10); axs[1].set_title("Cuánto acierta")
axs[1].grid(axis="x", visible=False)
plt.show()
puesto = list(pc.index[::-1]).index("latitud") + 1
display(Markdown(
    f"Con coordenadas acierta más, también con la validación por bloques. Pero la latitud es la "
    f"**variable número {puesto}** del modelo, y no sabe nada de foraminíferos. El modelo ha aprendido que "
    "cerca de los polos el agua es fría, y lo usa. Hoy eso es verdad. ¿Lo será también en el pasado?"))

### 6.2 Un sondeo que se enfría

En un sondeo, la latitud y la longitud son las mismas del fondo al techo: el sitio no se mueve en 20 000 años. Lo que cambia es la fauna.

Simulamos un sondeo frente a Lisboa que se enfría: le damos, una a una, las faunas de sitios del Atlántico Norte cada vez más fríos, sin tocar sus coordenadas. Y comparamos los dos modelos, entrenados sin esas faunas para que sean nuevas para ellos.

In [ ]:
#@title 6.2 La misma fauna, dos modelos
i0 = int(((datos["Latitude"] - 37.8) ** 2 + (datos["Longitude"] + 9.9) ** 2).idxmin())
atl = np.where((datos["Latitude"] > 30) & (datos["Longitude"].between(-75, 15)) & (y <= y[i0]))[0]
resto = np.setdiff1d(np.arange(len(X)), atl)          # los dos modelos se entrenan sin esas faunas
ms = nuevo_xgboost().fit(X.iloc[resto], y[resto])
mc2 = nuevo_xgboost().fit(XC.iloc[resto], y[resto])
fauna = X.iloc[atl]
p_solo = ms.predict(fauna)
p_coord = mc2.predict(fauna.assign(latitud=datos.loc[i0, "Latitude"], longitud=datos.loc[i0, "Longitude"]))
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.scatter(y[atl], p_solo, s=10, color=AZUL, alpha=0.6, lw=0, label="solo especies")
ax.scatter(y[atl], p_coord, s=10, color=ROJO, alpha=0.6, lw=0, label="especies + coordenadas de Lisboa")
ax.plot([-2, y[i0]], [-2, y[i0]], color=TINTA, lw=1, ls=(0, (4, 3)))
ax.axhline(y[i0], color=GRIS, lw=1, ls=(0, (2, 2)))
ax.text(-1.5, y[i0] + 0.4, f"hoy frente a Lisboa, {es(y[i0])} °C", color=GRIS, fontsize=10)
ax.set_xlabel("temperatura del sitio del que sale la fauna (°C)"); ax.set_ylabel("temperatura reconstruida (°C)")
ax.set_title("Un sondeo frente a Lisboa con faunas cada vez más frías"); ax.legend(loc="lower right", markerscale=3)
plt.show()
frio = y[atl] < 8
display(Markdown(
    f"Con faunas de agua por debajo de 8 °C, el modelo con solo especies reconstruye de media "
    f"**{es(p_solo[frio].mean())} °C** y el de coordenadas **{es(p_coord[frio].mean())} °C**, cuando la fauna "
    f"viene de agua a {es(y[atl][frio].mean())} °C. El modelo con coordenadas **se resiste a enfriar**, porque "
    "la latitud le sigue diciendo que está frente a Lisboa. Reconstruiría un pasado más parecido al presente "
    "de lo que fue."))

Es lo mismo que el husky y el lobo: el modelo acierta, pero mirando el fondo de la foto. Hoy el fondo y el animal van juntos; en el pasado, no. **Para eso sirve explicar un modelo: para pillarlo cuando acierta por malas razones**, antes de usarlo donde esas razones dejan de valer.

---
# 7. Los límites

Explicar un modelo no lo arregla. Hay algo que ningún modelo de estos sabe hacer bien: **predecir fuera de lo que ha visto**. En paleoclima pasa a menudo, porque el pasado tuvo climas que hoy no existen.

**Qué hacer:** elegir con el deslizador hasta qué temperatura ven los modelos al entrenar. Se entrenan solo con los sitios más fríos que ese límite y se les pide la temperatura de los más cálidos.

In [ ]:
#@title 7.1 Entrenar con el agua fría y predecir la cálida
limite = 20  #@param {type:"slider", min:12, max:26, step:1}
ent, pru = y < limite, y >= limite
MODS = {"Lineal": Ridge(alpha=1.0), "Random forest": RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=0),
        "XGBoost": nuevo_xgboost(), "k-NN (MAT)": KNeighborsRegressor(n_neighbors=5)}
fig, axs = plt.subplots(1, 4, figsize=(17, 4.4), sharey=True, gridspec_kw={"wspace": 0.08})
for ax, (nombre, m) in zip(axs, MODS.items()):
    p = m.fit(X[ent], y[ent]).predict(X)
    ax.scatter(y[ent], p[ent], s=4, color=GRIS, alpha=0.35, lw=0, label="entrenamiento")
    ax.scatter(y[pru], p[pru], s=5, color=ROJO, alpha=0.5, lw=0, label="más cálido que el límite")
    ax.plot([-4, 32], [-4, 32], color=TINTA, lw=1, ls=(0, (4, 3)))
    ax.axhline(limite, color=ROJO, lw=1, ls=(0, (2, 2)))
    ax.set_xlim(-4, 32); ax.set_ylim(-4, 32); ax.set_aspect("equal")
    ax.set_title(nombre); ax.set_xlabel("temperatura real (°C)")
    ax.text(0.04, 0.96, f"error en lo cálido\n{es(rmse(y[pru], p[pru]))} °C\nmáximo predicho {es(p[pru].max())} °C",
            transform=ax.transAxes, va="top", fontsize=11, color=ROJO)
axs[0].set_ylabel("temperatura predicha (°C)"); axs[0].legend(loc="lower right", fontsize=9, markerscale=3)
plt.show()
display(Markdown(
    f"Con {ent.sum()} sitios por debajo de {limite} °C para entrenar y {pru.sum()} por encima para probar. "
    "Los árboles y los vecinos **no pueden pasar del techo**: solo saben devolver valores que han visto. "
    "La recta sí sigue subiendo, pero a ciegas. Mueve el límite y mira cómo cambia el techo."))

Y lo que dice SHAP de esas predicciones también es del modelo: explicará con todo detalle por qué predice 20 °C en un sitio de 28 °C. **La explicación nunca es mejor que el modelo al que explica.**

---
# Chuleta para llevarse

**Las preguntas, en orden**

1. **¿Cuánto acierta?** Sin esto no hay nada que explicar. Validar como en la sesión 2.
2. **¿Qué variables usa?** El peso medio de cada una (SHAP en valor absoluto). ¿Son las que esperabas?
3. **¿Cómo actúa cada una?** La curva de efecto. ¿Tiene el sentido y la forma que dice lo que se sabe del sistema?
4. **¿Por qué ha dicho esto aquí?** La cascada de un caso concreto, sobre todo de los casos raros.
5. **¿Cuentan lo mismo otros modelos?** Si coinciden modelos distintos, la relación es de los datos.
6. **¿Hace trampa?** Variables que no deberían pesar tanto (la geografía, el número de muestra, la fecha).
7. **¿Le estoy pidiendo algo que no ha visto?** Fuera del rango de entrenamiento, ni el modelo ni su explicación valen.

**Avisos**

| Cuidado con | Por qué |
|---|---|
| Variables gemelas | Si dos dicen lo mismo, se reparten el mérito y las dos parecen menos importantes |
| Leer causas | SHAP dice qué usa el modelo, no qué causa qué en la naturaleza |
| Un solo modelo | Su explicación puede ser una manía suya: comparar con otros |
| Modelos malos | Si el modelo no acierta, explicarlo no sirve de nada |

**Para usarlo con datos propios.** Con modelos de árboles (random forest, XGBoost) SHAP es casi instantáneo: `shap.TreeExplainer(modelo).shap_values(X)`. Con cualquier otro modelo funciona `shap.Explainer(modelo.predict, X)`, solo que más despacio.

*Datos: ForCenS (Siccha y Kucera, 2017, Scientific Data 4, 170109) y World Ocean Atlas 2023 (NOAA). Mapa: Natural Earth. SHAP: Lundberg y Lee (2017), sobre los valores de Shapley (1953).*